# Historical Data — Schema Discovery

Discovers schema across DB1 (old production) and DB2 (production) for three domains:

1. **Sales** — sold items, carts/transactions, payment
2. **Orders & Manifests** — purchase orders, manifest headers/rows, CSV templates
3. **Inventory** — items, products, categories, product attributes

**Use cases:**
- 2025–2026 sales reporting
- Manifest header → standardized template mapping
- Category taxonomy + pricing intelligence for future order bids

**Run all cells, save, then share back for final export SQL.**

In [1]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, inspect, text

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


def _resolve_notebook_root() -> Path:
    """Find workspace/notebooks (parent of _shared/) from cwd or NOTEBOOK_DIR."""
    cwd = Path.cwd().resolve()
    if os.environ.get("NOTEBOOK_DIR"):
        p = Path(os.environ["NOTEBOOK_DIR"]).resolve()
        if (p / "_shared" / "config.example.py").is_file():
            return p
        if p.name == "_shared" and (p / "config.example.py").is_file():
            return p.parent
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / "workspace" / "notebooks"):
            if (candidate / "_shared" / "config.example.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not find workspace/notebooks/_shared/config.example.py; "
        "set NOTEBOOK_DIR to workspace/notebooks or run from repo root."
    )


NOTEBOOK_ROOT = _resolve_notebook_root()
SHARED_DIR = NOTEBOOK_ROOT / "_shared"
if str(SHARED_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_DIR))

_cfg = SHARED_DIR / "config_local.py"
if not _cfg.is_file():
    raise FileNotFoundError(
        f"Missing {_cfg} — copy _shared/config.example.py to _shared/config_local.py (see _shared/README.md)."
    )

from config_local import CONNECTIONS  # noqa: E402

OUTPUT_DIR = Path.cwd().resolve() if (Path.cwd().resolve() / "schema_discovery.ipynb").exists() else NOTEBOOK_ROOT / "historical-data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_engine(cfg: dict):
    user = quote_plus(str(cfg["user"]))
    pw = quote_plus(str(cfg["password"]))
    host = cfg["host"]
    port = int(cfg["port"])
    db = cfg["database"]
    return create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}", future=True)


def open_connection(key: str):
    cfg = CONNECTIONS[key]
    eng = make_engine(cfg)
    schema = str(cfg.get("schema") or "public")
    label = cfg.get("label", key)
    return eng, schema, label


def describe_table(engine, table_name, schema="public"):
    sql = text(
        "SELECT column_name, data_type, is_nullable, column_default "
        "FROM information_schema.columns "
        "WHERE table_schema = :schema AND table_name = :tbl "
        "ORDER BY ordinal_position"
    )
    with engine.connect() as conn:
        return pd.read_sql_query(sql, conn, params={"schema": schema, "tbl": table_name})


def sample_table(engine, table_name, schema="public", limit=5):
    sql = text(f'SELECT * FROM "{schema}"."{table_name}" LIMIT :lim')
    with engine.connect() as conn:
        return pd.read_sql_query(sql, conn, params={"lim": limit})


def row_count(engine, table_name, schema="public"):
    sql = text(f'SELECT COUNT(*) AS n FROM "{schema}"."{table_name}"')
    with engine.connect() as conn:
        return conn.execute(sql).scalar()


def table_exists(engine, table_name, schema="public"):
    return table_name in inspect(engine).get_table_names(schema=schema)


def get_fks(engine, table_name, schema="public"):
    sql = text("""
        SELECT kcu.column_name, ccu.table_name AS fk_table, ccu.column_name AS fk_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
        JOIN information_schema.constraint_column_usage ccu ON ccu.constraint_name = tc.constraint_name
        WHERE tc.table_schema = :schema AND tc.table_name = :tbl AND tc.constraint_type = 'FOREIGN KEY'
    """)
    with engine.connect() as conn:
        return pd.read_sql_query(sql, conn, params={"schema": schema, "tbl": table_name})


def full_describe(engine, table_name, schema="public", sample_n=5, show_fks=True):
    """One-shot: columns, row count, FKs, sample rows."""
    n = row_count(engine, table_name, schema)
    cols_df = describe_table(engine, table_name, schema)
    print(f"\n{'='*70}")
    print(f"  {table_name}  ({n:,} rows)")
    print(f"{'='*70}")
    print("\nColumns:")
    display(cols_df)
    if show_fks:
        fks = get_fks(engine, table_name, schema)
        if len(fks):
            print("\nForeign keys:")
            display(fks)
    if n > 0 and sample_n > 0:
        print(f"\nSample ({min(sample_n, n)} rows):")
        display(sample_table(engine, table_name, schema, sample_n))
    return cols_df, n


print("Notebook root:", NOTEBOOK_ROOT)
print("Shared dir:", SHARED_DIR)
print("Output dir:", OUTPUT_DIR)
print("Available DB keys:", list(CONNECTIONS.keys()))

Notebook dir: E:\ecothrift-dashboard\workspace\notebooks
Output dir: E:\ecothrift-dashboard\workspace\notebooks\historical-data
Available DB keys: ['old_production', 'production', 'dev']


---
# PART 1 — DB2 (Production)
2nd-generation app. Currently live. Tables prefixed `inventory_`, `pos_`, etc.

In [2]:
eng2, sch2, lab2 = open_connection("production")
print(f"Connected: {lab2}")

all_db2 = sorted(inspect(eng2).get_table_names(schema=sch2))
print(f"\nAll tables ({len(all_db2)}):")
for t in all_db2:
    print(f"  {t}")

Connected: DB2 — Production (local restore / Heroku name may differ)

All tables (84):
  auth_group
  auth_group_permissions
  auth_permission
  core_address
  core_business_event
  core_contact
  core_customer_profile
  core_employee_profile
  core_event_category
  core_manifest
  core_s3_file
  core_user
  core_user_groups
  core_user_user_permissions
  core_work_location
  django_admin_log
  django_cache_table
  django_content_type
  django_migrations
  django_session
  hr_availability
  hr_blackout_date
  hr_budget
  hr_clock_event
  hr_compensation
  hr_department
  hr_holiday
  hr_holiday_departments
  hr_onboarding_checklist
  hr_pay_grade
  hr_pay_period
  hr_permission_audit_log
  hr_permission_profile
  hr_pto_accrual
  hr_pto_balance
  hr_pto_policy
  hr_pto_request
  hr_schedule
  hr_shift
  hr_time_audit_log
  hr_time_break
  hr_time_entry
  hr_time_modification_request
  hr_time_notification
  hr_time_notification_preference
  inventory_csv_field_mapping
  inventory_csv_t

## 1A — DB2 Inventory tables

In [3]:
db2_inv_tables = [t for t in all_db2 if t.startswith("inventory_")]
print("Inventory tables:", db2_inv_tables)

db2_schemas = {}
for t in db2_inv_tables:
    cols_df, n = full_describe(eng2, t, sch2, sample_n=3)
    db2_schemas[t] = {"cols": set(cols_df["column_name"]), "n": n}

Inventory tables: ['inventory_csv_field_mapping', 'inventory_csv_template', 'inventory_customer_message', 'inventory_item', 'inventory_item_history', 'inventory_item_scan_history', 'inventory_location', 'inventory_manifest_rows', 'inventory_preprocessing_detail', 'inventory_pricing_template', 'inventory_processing_detail', 'inventory_product', 'inventory_product_class', 'inventory_purchase_order', 'inventory_raw_manifest', 'inventory_standardization_run', 'inventory_store_configuration', 'inventory_vendor']

  inventory_csv_field_mapping  (120 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,None
1,source_column,character varying,NO,None
2,target_field,character varying,NO,None
3,transform_expression,text,NO,None
4,default_value,character varying,NO,None
5,is_required,boolean,NO,None
6,order,integer,NO,None
7,template_id,bigint,NO,None



Foreign keys:


,column_name,fk_table,fk_column
0,template_id,inventory_csv_template,id



Sample (3 rows):


...

,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,None
1,name,character varying,NO,None
2,code,character varying,NO,None
3,vendor_type,character varying,NO,None
4,contact_name,character varying,NO,None
5,email,character varying,NO,None
6,phone,character varying,NO,None
7,website,character varying,NO,None
8,address_line1,character varying,NO,None
9,address_line2,character varying,NO,None



Foreign keys:


,column_name,fk_table,fk_column
0,created_by_id,core_user,id



Sample (3 rows):


,id,name,code,vendor_type,contact_name,email,phone,website,address_line1,address_line2,city,state,zip_code,created_at,updated_at,created_by_id
0,1,Walmart,WAL,liquidation,,,,https://liquidations.walmart.com/,,,,,,2025-08-18 17:54:20.384821+00:00,2025-08-18 17:54:20.384836+00:00,1
1,2,Target,TRGET,liquidation,,,,https://bstock.com/buy/seller/target,,,,,,2025-08-25 14:46:58.619332+00:00,2025-08-25 14:46:58.619347+00:00,1
2,35,Costco,CST,liquidation,,,,https://bstock.com/costco,,,,,,2025-08-29 14:01:52.779465+00:00,2025-08-29 14:01:52.779479+00:00,1


## 1B — DB2 POS / Sales tables

In [4]:
db2_pos_tables = [t for t in all_db2 if t.startswith("pos_") or "cart" in t or "sale" in t or "transaction" in t or "payment" in t]
print("POS/Sales tables:", db2_pos_tables)

for t in db2_pos_tables:
    cols_df, n = full_describe(eng2, t, sch2, sample_n=3)
    db2_schemas[t] = {"cols": set(cols_df["column_name"]), "n": n}

POS/Sales tables: ['pos_bank_transaction', 'pos_cart', 'pos_cart_line', 'pos_cash_deposit', 'pos_cash_management_settings', 'pos_change_request', 'pos_discount_rule', 'pos_drawer', 'pos_drawer_shift', 'pos_payment', 'pos_receipt', 'pos_receipt_template', 'pos_register', 'pos_revenue_goal', 'pos_supplemental_drawer', 'pos_supplemental_drawer_transaction', 'pos_transaction_void']

  pos_bank_transaction  (12 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,None
1,transaction_date,date,NO,None
2,transaction_type,character varying,NO,None
3,deposit_amount,numeric,NO,None
4,change_amount,numeric,NO,None
5,bank_confirmation_number,character varying,NO,None
6,notes,text,NO,None
7,created_at,timestamp with time zone,NO,None
8,performed_by_id,bigint,YES,None
9,work_location_id,bigint,NO,None



Foreign keys:


,column_name,fk_table,fk_column
0,performed_by_id,core_user,id
1,work_location_id,core_work_location,id



Sample (3 rows):


...

,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,None
1,void_type,character varying,NO,None
2,manager_pin,character varying,NO,None
3,reason,text,NO,None
4,refund_amount,numeric,NO,None
5,items_returned,jsonb,NO,None
6,approved,boolean,NO,None
7,completed,boolean,NO,None
8,created_at,timestamp with time zone,NO,None
9,completed_at,timestamp with time zone,YES,None



Foreign keys:


,column_name,fk_table,fk_column
0,approved_by_id,core_user,id
1,new_cart_id,pos_cart,id
2,original_cart_id,pos_cart,id
3,requested_by_id,core_user,id



Sample (3 rows):


,id,void_type,manager_pin,reason,refund_amount,items_returned,approved,completed,created_at,completed_at,approved_by_id,new_cart_id,original_cart_id,requested_by_id
0,1,void,1234,TESTING,0.0,[],True,True,2025-08-23 13:58:39.102843-05:00,2025-08-23 18:58:39.174177+00:00,3.0,None,5,3
1,2,void,1234,TESTING,0.0,[],True,True,2025-08-23 13:58:48.257973-05:00,2025-08-23 18:58:48.317190+00:00,3.0,None,4,3
2,34,void,6661,i hate it,0.0,[],False,False,2026-01-21 14:31:19.594376-06:00,NaT,NaN,None,14200,23


## 1C — DB2 Other potentially relevant tables
Anything with order, manifest, vendor, csv, template, category in the name that wasn't already covered.

In [5]:
already = set(db2_inv_tables + db2_pos_tables)
keywords = ["order", "manifest", "vendor", "csv", "template", "categ", "core_"]
db2_other = [t for t in all_db2 if t not in already and any(k in t.lower() for k in keywords)]
print("Other relevant tables:", db2_other)

for t in db2_other:
    cols_df, n = full_describe(eng2, t, sch2, sample_n=3)
    db2_schemas[t] = {"cols": set(cols_df["column_name"]), "n": n}

Other relevant tables: ['core_address', 'core_business_event', 'core_contact', 'core_customer_profile', 'core_employee_profile', 'core_event_category', 'core_manifest', 'core_s3_file', 'core_user', 'core_user_groups', 'core_user_user_permissions', 'core_work_location']

  core_address  (23 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,None
1,object_id,integer,YES,None
2,address_type,character varying,NO,None
3,address_line_1,character varying,NO,None
4,address_line_2,character varying,NO,None
5,city,character varying,NO,None
6,state,character varying,NO,None
7,zip_code,character varying,NO,None
8,country,character varying,NO,None
9,created_at,timestamp with time zone,NO,None



Foreign keys:


,column_name,fk_table,fk_column
0,content_type_id,django_content_type,id



Sample (3 rows):


...

,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,None
1,name,character varying,NO,None
2,code,character varying,NO,None
3,location_type,character varying,NO,None
4,is_active,boolean,NO,None
5,created_at,timestamp with time zone,NO,None
6,updated_at,timestamp with time zone,NO,None
7,address_id,bigint,NO,None
8,theme_color,character varying,NO,None
9,gradient_start,character varying,NO,None



Foreign keys:


,column_name,fk_table,fk_column
0,address_id,core_address,id



Sample (3 rows):


,id,name,code,location_type,is_active,created_at,updated_at,address_id,theme_color,gradient_start,gradient_end
0,1,Warehouse 1,WH1,warehouse,True,2025-08-19 21:14:23.831337+00:00,2025-08-19 21:14:23.831352+00:00,1,#000000,#000000,#000000
1,2,Warehouse 2,WH2,warehouse,True,2025-08-19 21:14:23.839698+00:00,2025-08-19 21:14:23.839710+00:00,2,#000000,#000000,#000000
2,3,Corporate Office,CORP,office,True,2025-08-19 21:14:23.846495+00:00,2025-08-19 21:14:23.846502+00:00,3,#000000,#000000,#000000


## 1D — DB2 Date ranges & counts

In [6]:
db2_stats_sql = text("""
SELECT
    -- Items
    (SELECT COUNT(*) FROM inventory_item)                                           AS total_items,
    (SELECT COUNT(*) FROM inventory_item WHERE sold_at IS NOT NULL)                 AS sold_items,
    (SELECT COUNT(*) FROM inventory_item WHERE sold_at >= '2025-01-01')             AS sold_since_2025,
    (SELECT COUNT(*) FROM inventory_item WHERE sold_at >= NOW() - INTERVAL '16 months') AS sold_last_16mo,
    (SELECT MIN(sold_at) FROM inventory_item WHERE sold_at IS NOT NULL)             AS earliest_sale,
    (SELECT MAX(sold_at) FROM inventory_item WHERE sold_at IS NOT NULL)             AS latest_sale,
    -- Products
    (SELECT COUNT(*) FROM inventory_product)                                        AS total_products,
    -- Date range of all items
    (SELECT MIN(created_on) FROM inventory_item)                                    AS earliest_item_created,
    (SELECT MAX(created_on) FROM inventory_item)                                    AS latest_item_created
""")

# created_on might not exist — fall back to id ordering
try:
    with eng2.connect() as conn:
        df_db2_stats = pd.read_sql_query(db2_stats_sql, conn)
    print("=== DB2 Key Stats ===")
    display(df_db2_stats.T.rename(columns={0: "value"}))
except Exception as e:
    print(f"Stats query error (column name issue?): {e}")
    # Simpler fallback
    fallback = text("""
    SELECT
        (SELECT COUNT(*) FROM inventory_item) AS total_items,
        (SELECT COUNT(*) FROM inventory_item WHERE sold_at IS NOT NULL) AS sold_items,
        (SELECT COUNT(*) FROM inventory_item WHERE sold_at >= '2025-01-01') AS sold_since_2025,
        (SELECT COUNT(*) FROM inventory_item WHERE sold_at >= NOW() - INTERVAL '16 months') AS sold_last_16mo,
        (SELECT MIN(sold_at) FROM inventory_item WHERE sold_at IS NOT NULL) AS earliest_sale,
        (SELECT MAX(sold_at) FROM inventory_item WHERE sold_at IS NOT NULL) AS latest_sale,
        (SELECT COUNT(*) FROM inventory_product) AS total_products
    """)
    with eng2.connect() as conn:
        df_db2_stats = pd.read_sql_query(fallback, conn)
    print("=== DB2 Key Stats (fallback) ===")
    display(df_db2_stats.T.rename(columns={0: "value"}))

Stats query error (column name issue?): Execution failed on sql '
SELECT
    -- Items
    (SELECT COUNT(*) FROM inventory_item)                                           AS total_items,
    (SELECT COUNT(*) FROM inventory_item WHERE sold_at IS NOT NULL)                 AS sold_items,
    (SELECT COUNT(*) FROM inventory_item WHERE sold_at >= '2025-01-01')             AS sold_since_2025,
    (SELECT COUNT(*) FROM inventory_item WHERE sold_at >= NOW() - INTERVAL '16 months') AS sold_last_16mo,
    (SELECT MIN(sold_at) FROM inventory_item WHERE sold_at IS NOT NULL)             AS earliest_sale,
    (SELECT MAX(sold_at) FROM inventory_item WHERE sold_at IS NOT NULL)             AS latest_sale,
    -- Products
    (SELECT COUNT(*) FROM inventory_product)                                        AS total_products,
    -- Date range of all items
    (SELECT MIN(created_on) FROM inventory_item)                                    AS earliest_item_created,
    (SELECT MAX(created_on) FROM inventory

,value
total_items,59833
sold_items,34762
sold_since_2025,34762
sold_last_16mo,34762
earliest_sale,2025-08-23 15:11:11.629839+00:00
latest_sale,2026-03-04 18:16:58.698128+00:00
total_products,41509


## 1E — DB2 Category deep-dive
Shows all category-related columns on product, plus the full `inventory_category` table if it exists.

In [7]:
ip_cols = db2_schemas.get("inventory_product", {}).get("cols", set())
cat_cols_db2 = sorted(c for c in ip_cols if any(k in c.lower() for k in ["categ", "sub", "type", "class", "dept"]))
print("inventory_product category-related columns:", cat_cols_db2)

for col in cat_cols_db2:
    sql = text(f'SELECT "{col}"::text AS val, COUNT(*) AS cnt FROM inventory_product GROUP BY 1 ORDER BY cnt DESC LIMIT 40')
    with eng2.connect() as conn:
        df_vals = pd.read_sql_query(sql, conn)
    print(f"\nDistinct values — inventory_product.{col} (top 40):")
    display(df_vals)

if table_exists(eng2, "inventory_category", sch2):
    sql_all_cats = text("SELECT * FROM inventory_category ORDER BY id")
    with eng2.connect() as conn:
        df_cats = pd.read_sql_query(sql_all_cats, conn)
    print(f"\nFull inventory_category table ({len(df_cats)} rows):")
    display(df_cats)
else:
    print("\ninventory_category table does not exist in DB2.")

inventory_product category-related columns: ['product_class_id']

Distinct values — inventory_product.product_class_id (top 40):


,val,cnt
0,None,41509



inventory_category table does not exist in DB2.


## 1F — DB2 Purchase Order stats

In [8]:
po_table = None
for candidate in ["inventory_purchaseorder", "inventory_purchase_order", "purchase_order"]:
    if candidate in db2_schemas:
        po_table = candidate
        break

if po_table:
    po_cols = db2_schemas[po_table]["cols"]
    # Try to find date columns
    date_cols = sorted(c for c in po_cols if any(k in c.lower() for k in ["date", "created", "ordered", "time"]))
    print(f"PO table: {po_table} ({db2_schemas[po_table]['n']} rows)")
    print(f"Date columns: {date_cols}")
    # Show distinct statuses if status column exists
    for col in ["status", "order_status"]:
        if col in po_cols:
            sql = text(f'SELECT "{col}", COUNT(*) AS cnt FROM "{po_table}" GROUP BY 1 ORDER BY cnt DESC')
            with eng2.connect() as conn:
                display(pd.read_sql_query(sql, conn))
else:
    print("No purchase order table found in DB2.")

# Manifest / CSV template tables
for t in ["inventory_manifest", "inventory_manifestrow", "inventory_csv_template", "inventory_csvtemplate",
          "inventory_csv_field_mapping", "inventory_csvfieldmapping", "core_manifest"]:
    if t in db2_schemas:
        print(f"\n{t}: {db2_schemas[t]['n']} rows")

PO table: inventory_purchase_order (103 rows)
Date columns: ['created_at', 'created_by_id', 'purchase_date', 'received_date', 'updated_at']


,status,cnt
0,items_generated,62
1,received,40
2,confirmed,1



inventory_csv_template: 10 rows

inventory_csv_field_mapping: 120 rows

core_manifest: 0 rows


---
# PART 2 — DB1 (Old Production)
1st-generation app. Tables are unprefixed: `item`, `product`, `cart`, `cart_line`, `purchase_order`, `manifest`, etc.

In [9]:
eng1, sch1, lab1 = open_connection("old_production")
print(f"Connected: {lab1}")

all_db1 = sorted(inspect(eng1).get_table_names(schema=sch1))
print(f"\nAll tables ({len(all_db1)}):")
for t in all_db1:
    print(f"  {t}")

Connected: DB1 — Old Production (archive)

All tables (58):
  accounts_profile
  address
  area
  auth_group
  auth_group_permissions
  auth_permission
  auth_user
  auth_user_groups
  auth_user_user_permissions
  cart
  cart_discount
  cart_line
  data_value
  department
  department_budget
  discount
  django_admin_log
  django_content_type
  django_migrations
  django_session
  drawer
  ecothrift_holidays
  employee
  employee_availability
  employee_compensation
  employee_departments
  giftcard
  item
  item_condition
  item_destination
  item_dispatch
  item_location
  item_status
  item_test_result
  list_condition
  list_status
  location
  manifest
  pay_period
  pay_week
  payment
  payroll_history
  permissions
  person
  price
  price_index
  product
  product_attrs
  purchase_order
  requests_off
  schedule
  shift
  standard_bogo
  thrift_plus
  thrift_plus_payment
  timeclock
  timeclock_entry
  v_price_label


## 2A — DB1 Inventory tables
`item`, `product`, `product_attrs`, and anything with categ/condition/status/location in the name.

In [10]:
inv_keywords = ["item", "product", "categ", "condition", "status", "location", "price", "manifest"]
db1_inv_tables = [t for t in all_db1 if any(k in t.lower() for k in inv_keywords)]
print("Inventory-related tables:", db1_inv_tables)

db1_schemas = {}
for t in db1_inv_tables:
    cols_df, n = full_describe(eng1, t, sch1, sample_n=3)
    db1_schemas[t] = {"cols": set(cols_df["column_name"]), "n": n}

Inventory-related tables: ['item', 'item_condition', 'item_destination', 'item_dispatch', 'item_location', 'item_status', 'item_test_result', 'list_condition', 'list_status', 'location', 'manifest', 'price', 'price_index', 'product', 'product_attrs', 'v_price_label']

  item  (123,941 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,integer,NO,nextval('item_id_seq'::regclass)
1,code,character,YES,create_table_code('item'::text)
2,bulk_cde,character,YES,NaN
3,quantity,integer,YES,1
4,order_number,text,YES,NaN
5,line_number,integer,YES,NaN
6,product_cde,character,YES,NaN
7,price_lbl,character varying,YES,NaN
8,retail_amt,numeric,YES,NaN
9,starting_price_amt,numeric,YES,NaN



Sample (3 rows):


,id,code,bulk_cde,quantity,order_number,line_number,product_cde,price_lbl,retail_amt,starting_price_amt,is_static
0,586,8LvknXmkx,,1,TGT100653,136,SMipQ17AY,Z6,35.00,26.25,False
1,13,jNZeWeGtR,NaN,1,AMZ11175,13,xwzLvOj3L,NaN,38.84,36.90,False
2,25,0LkGzcZDG,NaN,1,AMZ11175,25,ndCCDyLLn,A1,34.99,33.24,False



  item_condition  (130,112 rows)

Columns:


...

,id,product_cde,upc,category,subcategory,retail_amt,attrs,quantity,last_instance,taxable
0,1,mksNtyUkK,8939865242886,Home & Decor,Kitchenware,169.95,None,1,2024-03-14 13:01:52.825979+00:00,True
1,2,mKX9OKzMV,61124738833,Home & Decor,Appliances,89.00,None,1,2024-03-14 13:01:52.825979+00:00,True
2,3,QZCKFL04a,6285435218,Home & Decor,Kitchenware,84.38,None,1,2024-03-14 13:01:52.825979+00:00,True



  v_price_label  (190 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,price_dynamic_lbl,text,YES,None



Sample (3 rows):


,price_dynamic_lbl
0,U9
1,V1
2,V2


## 2B — DB1 Sales tables
`cart`, `cart_line`, `payment`, `discount`, `giftcard`, `drawer`.

In [11]:
sales_keywords = ["cart", "payment", "discount", "giftcard", "drawer", "sale", "transaction"]
db1_sales_tables = [t for t in all_db1 if any(k in t.lower() for k in sales_keywords) and t not in db1_schemas]
print("Sales-related tables:", db1_sales_tables)

for t in db1_sales_tables:
    cols_df, n = full_describe(eng1, t, sch1, sample_n=3)
    db1_schemas[t] = {"cols": set(cols_df["column_name"]), "n": n}

Sales-related tables: ['cart', 'cart_discount', 'cart_line', 'discount', 'drawer', 'giftcard', 'payment', 'thrift_plus_payment']

  cart  (53,304 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,integer,NO,nextval('cart_id_seq'::regclass)
1,code,character varying,YES,NaN
2,drawer_cde,character varying,YES,NaN
3,open_time,timestamp with time zone,YES,NaN
4,close_time,timestamp with time zone,YES,NaN
5,subtotal_amt,numeric,YES,NaN
6,total_discount_amt,numeric,YES,NaN
7,total_taxable_amt,numeric,YES,NaN
8,sales_tax_percentage,numeric,YES,NaN
9,tax_amt,numeric,YES,NaN



Sample (3 rows):


,id,code,drawer_cde,open_time,close_time,subtotal_amt,total_discount_amt,total_taxable_amt,sales_tax_percentage,tax_amt,total_amt,total_payment_amt,notes,void,void_reason,replaced_cart_cde,thrift_plus_member_id
0,34265,1RB3U2OQj,w3FzPYBHi,2024-02-25 10:20:45-06:00,2024-02-25 10:39:59-06:00,11.0,0.0,11.0,7.0,0.77,11.77,20.0,,False,,None,None
1,31176,lsI8SvUet,yLiDObHf0,2023-12-15 07:29:30-06:00,2023-12-15 07:31:09-06:00,10.0,0.0,10.0,7.0,0.70,10.70,10.7,,False,,None,None
2,29486,43f67j1Cb,nfzCX4Nvv,2023-09-27 09:07:39-05:00,2023-09-27 09:26:48-05:00,4.0,0.0,4.0,7.0,0.28,4.28,4.5,NaN,False,NaN,None,None



  cart_discount  (1,505 rows)

Columns:


...

,column_name,data_type,is_nullable,column_default
0,id,integer,NO,nextval('thrift_plus_payment_id_seq'::regclass)
1,member_id,integer,NO,NaN
2,cart_id,integer,YES,NaN
3,payment_date,date,YES,CURRENT_DATE
4,amount,numeric,YES,NaN
5,duration_months,integer,YES,NaN
6,start_date,date,YES,NaN
7,end_date,date,YES,NaN



Foreign keys:


,column_name,fk_table,fk_column
0,cart_id,cart,id
1,member_id,thrift_plus,member_id



Sample (3 rows):


,id,member_id,cart_id,payment_date,amount,duration_months,start_date,end_date
0,1,307300189,51552,2024-10-29,19.99,1,2024-10-29,2024-11-28
1,2,901761238,51553,2024-10-29,19.99,1,2024-10-29,2024-11-28
2,4,668738426,51609,2024-10-30,19.99,1,2024-10-30,2024-11-29


## 2C — DB1 Order / Vendor tables

In [12]:
order_keywords = ["order", "vendor", "purchase", "department"]
db1_order_tables = [t for t in all_db1 if any(k in t.lower() for k in order_keywords) and t not in db1_schemas]
print("Order-related tables:", db1_order_tables)

for t in db1_order_tables:
    cols_df, n = full_describe(eng1, t, sch1, sample_n=3)
    db1_schemas[t] = {"cols": set(cols_df["column_name"]), "n": n}

Order-related tables: ['department', 'department_budget', 'employee_departments', 'purchase_order']

  department  (8 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,integer,NO,nextval('departments_department_id_seq'::regclass)
1,name,character varying,NO,NaN
2,manager_id,integer,YES,NaN



Foreign keys:


,column_name,fk_table,fk_column
0,manager_id,employee,id



Sample (3 rows):


...

,employee_id,department_id
0,79,5
1,82,4
2,9,4



  purchase_order  (210 rows)

Columns:


,column_name,data_type,is_nullable,column_default
0,id,integer,NO,nextval('purchase_order_id_seq1'::regclass)
1,number,text,YES,NaN
2,created_on,timestamp with time zone,YES,get_time()
3,retail_amt,numeric,YES,NaN
4,condition_id,integer,YES,NaN
5,quantity,integer,YES,NaN
6,description,text,YES,NaN
7,price_amt,numeric,YES,NaN
8,fee_amt,numeric,YES,NaN
9,shipping_amt,numeric,YES,NaN



Sample (3 rows):


,id,number,created_on,retail_amt,condition_id,quantity,description,price_amt,fee_amt,shipping_amt,paid_amt,delivery_address_cde,scheduled_delivery,received_on,purchased_on,paid_on,preprocessed_on,processed_on
0,277,HMD64501,2025-04-29 00:00:00+00:00,93414.0,4,1348,"Truckload of Tools & Hardware, Stores Inventory, 1,348 Units, Ext. Retail $93,414, Atlanta, GA",17575.0,703.0,2225.95,20503.95,WTx5czCwn,2025-05-06 22:00:00+00:00,2025-05-05 17:00:00+00:00,2025-04-15 17:00:00+00:00,2025-04-22 17:00:00+00:00,2025-04-29 17:00:00+00:00,None
1,323,TGT127191,2025-06-17 15:10:15.455153+00:00,61371.0,4,1478,"Truckload (24 Pallets) of Bedding, Soft Décor, Lighting/Wall Décor & More, Used - Good Condition, 1,478 Units, Ext. ...",6950.0,278.0,1441.61,8669.61,WTx5czCwn,2025-06-19 13:00:00+00:00,2025-06-18 17:00:00+00:00,2025-06-04 17:00:00+00:00,2025-06-07 17:00:00+00:00,2025-06-08 17:00:00+00:00,None
2,327,WFR12437,2025-06-17 15:18:17.514587+00:00,73297.0,5,101,"Truckload of Bedroom, Outdoor, Kitchen & Dining Furniture & More, 101 Units, Retail $73,297, Used - Fair Condition, ...",4025.0,161.0,1328.74,5514.74,WTx5czCwn,2025-06-19 22:00:00+00:00,2025-06-19 17:00:00+00:00,2025-06-11 17:00:00+00:00,2025-06-12 17:00:00+00:00,2025-06-12 17:00:00+00:00,None


## 2D — DB1 Category deep-dive

In [13]:
# Product table categories
p_cols = db1_schemas.get("product", {}).get("cols", set())
cat_cols_db1 = sorted(c for c in p_cols if any(k in c.lower() for k in ["categ", "sub", "type", "class", "dept"]))
print("product category-related columns:", cat_cols_db1)

for col in cat_cols_db1:
    sql = text(f'SELECT "{col}"::text AS val, COUNT(*) AS cnt FROM product GROUP BY 1 ORDER BY cnt DESC LIMIT 40')
    with eng1.connect() as conn:
        df_vals = pd.read_sql_query(sql, conn)
    print(f"\nDistinct values — product.{col} (top 40):")
    display(df_vals)

# product_attrs categories
if "product_attrs" in db1_schemas:
    pa_cols = db1_schemas["product_attrs"]["cols"]
    cat_in_pa = sorted(c for c in pa_cols if any(k in c.lower() for k in ["categ", "sub", "type", "class", "dept"]))
    print(f"\nproduct_attrs category-related columns: {cat_in_pa}")
    for col in cat_in_pa:
        sql = text(f'SELECT "{col}"::text AS val, COUNT(*) AS cnt FROM product_attrs GROUP BY 1 ORDER BY cnt DESC LIMIT 40')
        with eng1.connect() as conn:
            df_vals = pd.read_sql_query(sql, conn)
        print(f"\nDistinct values — product_attrs.{col} (top 40):")
        display(df_vals)

product category-related columns: []

product_attrs category-related columns: ['category', 'subcategory']

Distinct values — product_attrs.category (top 40):


,val,cnt
0,Home & Decor,62805
1,"Toys, Games & Arts",23461
2,Home Improvement & Tools,11169
3,Pets & Animal Care,9466
4,"Beauty, Health & Personal Care",8521
5,Electronics & Technology,8293
6,"Sports, Fitness & Outdoors",8123
7,"Clothing, Shoes & Accessories",5257
8,Media & Entertainment,3959
9,Office & School Supplies,3099



Distinct values — product_attrs.subcategory (top 40):


,val,cnt
0,Home Decor,13905
1,Kitchenware,13339
2,Bedding & Bath,13177
3,Storage & Organization,10989
4,Dolls & Action Figures,9039
5,Dogs,6847
6,Appliances,4630
7,Furniture,4289
8,Games & Puzzles,3704
9,Phones & Accessories,3439


## 2E — DB1 Date ranges & counts

In [14]:
db1_stats_sql = text("""
SELECT
    (SELECT COUNT(*) FROM item)                                                              AS total_items,
    (SELECT COUNT(DISTINCT i.code)
     FROM item i JOIN cart_line cl ON cl.item_cde = i.code
     JOIN cart c ON c.code = cl.cart_cde
     WHERE c.close_time IS NOT NULL AND c.void = false)                                      AS distinct_items_sold,
    (SELECT COUNT(*)
     FROM cart_line cl JOIN cart c ON c.code = cl.cart_cde
     WHERE c.close_time IS NOT NULL AND c.void = false)                                      AS total_sale_lines,
    (SELECT COUNT(*)
     FROM cart_line cl JOIN cart c ON c.code = cl.cart_cde
     WHERE c.close_time IS NOT NULL AND c.void = false
       AND c.close_time >= '2025-01-01')                                                     AS sale_lines_since_2025,
    (SELECT COUNT(*)
     FROM cart_line cl JOIN cart c ON c.code = cl.cart_cde
     WHERE c.close_time IS NOT NULL AND c.void = false
       AND c.close_time >= NOW() - INTERVAL '16 months')                                     AS sale_lines_last_16mo,
    (SELECT MIN(c.close_time) FROM cart c WHERE c.close_time IS NOT NULL AND c.void = false) AS earliest_sale,
    (SELECT MAX(c.close_time) FROM cart c WHERE c.close_time IS NOT NULL AND c.void = false) AS latest_sale,
    (SELECT COUNT(*) FROM product)                                                           AS total_products
""")

try:
    with eng1.connect() as conn:
        df_db1_stats = pd.read_sql_query(db1_stats_sql, conn)
    print("=== DB1 Key Stats ===")
    display(df_db1_stats.T.rename(columns={0: "value"}))
except Exception as e:
    print(f"Stats query error: {e}")
    print("Check column names against the item/cart schema above.")

=== DB1 Key Stats ===


,value
total_items,123941
distinct_items_sold,64058
total_sale_lines,172868
sale_lines_since_2025,78460
sale_lines_last_16mo,90016
earliest_sale,2023-02-21 12:28:16+00:00
latest_sale,9999-12-31 00:00:00+00:00
total_products,140621


---
# PART 3 — Summary

Compact overview of everything discovered.

In [15]:
print("=" * 70)
print("  SCHEMA DISCOVERY SUMMARY")
print("=" * 70)

print("\n--- DB2 (Production) ---")
print(f"Tables found: {len(all_db2)}")
for t, info in sorted(db2_schemas.items()):
    print(f"  {t:45s}  {info['n']:>8,} rows   cols: {sorted(info['cols'])}")

print(f"\n--- DB1 (Old Production) ---")
print(f"Tables found: {len(all_db1)}")
for t, info in sorted(db1_schemas.items()):
    print(f"  {t:45s}  {info['n']:>8,} rows   cols: {sorted(info['cols'])}")

print("\n" + "=" * 70)
print("SAVE THIS NOTEBOOK WITH OUTPUTS.")
print("Share back for final export query generation.")
print("=" * 70)

  SCHEMA DISCOVERY SUMMARY

--- DB2 (Production) ---
Tables found: 84
  core_address                                         23 rows   cols: ['address_line_1', 'address_line_2', 'address_type', 'city', 'content_type_id', 'country', 'created_at', 'id', 'object_id', 'state', 'updated_at', 'zip_code']
  core_business_event                                   0 rows   cols: ['actor_id', 'category_id', 'content_type_id', 'created_at', 'description', 'event_type', 'id', 'ip_address', 'level', 'metadata', 'object_id', 'target_user_id', 'user_agent']
  core_contact                                          0 rows   cols: ['contact_type', 'contact_value', 'content_type_id', 'created_at', 'id', 'is_primary', 'label', 'object_id', 'updated_at']
  core_customer_profile                                 0 rows   cols: ['created_at', 'customer_number', 'customer_since', 'email_opt_in', 'id', 'lifetime_spent', 'loyalty_tier', 'sms_opt_in', 'updated_at', 'user_id']
  core_employee_profile                  

---
## Cleanup

In [16]:
eng2.dispose()
eng1.dispose()
print("Both engines disposed.")

Both engines disposed.
